<a href="https://colab.research.google.com/github/ekomissarov/demos/blob/main/sql_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports, run to get all the magic

In [ ]:
# Python libs
import pandas as pd

# Magics
from helpers import (
    load_sql_magic,
    load_viz_magic
)
load_sql_magic()          # %%sql   — query DataFrames via duckdb (no extra installs)
load_viz_magic()          # %%viz   — open a DataFrame in PyGWalker (drag-and-drop charts)

True

# Case - gaps and islands pattern

In [ ]:
import pandas as pd
import numpy as np

# Генерируем синтетические данные активности пользователей
# np.random.seed(42)

data = []
# Пользователь 1: Длинные стрики с небольшими пропускaми
dates_u1 = pd.date_range('2026-07-01', '2026-07-10').tolist() + \
           pd.date_range('2026-07-15', '2026-07-20').tolist() + \
           pd.date_range('2026-07-21', '2026-07-25').tolist() # обратите внимание, 20 и 21 стыкуются (один остров)

# Пользователь 2: Короткие частые стрики
dates_u2 = pd.date_range('2026-07-02', '2026-07-04').tolist() + \
           pd.date_range('2026-07-08', '2026-07-09').tolist() + \
           pd.date_range('2026-07-15', '2026-07-18').tolist()

# Пользователь 3: Одиночные заходы и один длинный стрик
dates_u3 = [pd.Timestamp('2026-07-01'), pd.Timestamp('2026-07-05')] + \
           pd.date_range('2026-07-10', '2026-07-16').tolist()

for d in dates_u1:
    data.append({'user_id': 101, 'activity_date': d.date()})
for d in dates_u2:
    data.append({'user_id': 102, 'activity_date': d.date()})
for d in dates_u3:
    data.append({'user_id': 103, 'activity_date': d.date()})

# Создаем Pandas DataFrame
user_activity = pd.DataFrame(data)

# Проверим первые строки
user_activity.head(10)

,user_id,activity_date
0,101,2026-07-01
1,101,2026-07-02
2,101,2026-07-03
3,101,2026-07-04
4,101,2026-07-05
5,101,2026-07-06
6,101,2026-07-07
7,101,2026-07-08
8,101,2026-07-09
9,101,2026-07-10


In [ ]:
%%sql result <<

WITH

unique_periods AS (
-- Оставляем только уникальные моменты времени с точностью дескретизации острова (в данном случае день)
-- и присваиваем каждому dt его порядковый номер.
    SELECT
        *
        , ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY dt) as rn
    FROM (
        SELECT DISTINCT  -- на случай если могут происходить разные события
            user_id
            , DATE_TRUNC('day', activity_date) AS dt
        FROM user_activity
    )
),

session_groups AS (
-- Для каждого дня вычисляем "ключ группы".
-- У последовательных дней значение (dt - rn * 1 day)
-- будет одинаковым, что позволяет объединить их в непрерывные интервалы.

    SELECT
        user_id
        , dt
        , dt - rn * INTERVAL '1 day' as grp
    FROM unique_periods
)

-- Для каждой непрерывной последовательности определяем её начало и конец.
SELECT
    user_id
    --, grp
    , MIN(dt) as session_start
    , MAX(dt) as session_end
FROM session_groups
GROUP BY user_id, grp

-- если нужна фильтрация интервалов по длине
HAVING COUNT(*) > 1

-- Сортируем интервалы по времени начала.
ORDER BY user_id ASC, session_start ASC;

,user_id,session_start,session_end
0,101,2026-07-01,2026-07-10
1,101,2026-07-15,2026-07-25
2,102,2026-07-02,2026-07-04
3,102,2026-07-08,2026-07-09
4,102,2026-07-15,2026-07-18
5,103,2026-07-10,2026-07-16


# Case - gaps and islands pattern (longest sequence)

In [ ]:
import pandas as pd
import numpy as np

# Генерируем синтетические данные активности пользователей
# np.random.seed(42)

data = []
# Пользователь 1: Длинные стрики с небольшими пропускaми
dates_u1 = pd.date_range('2026-07-01', '2026-07-10').tolist() + \
           pd.date_range('2026-07-15', '2026-07-20').tolist() + \
           pd.date_range('2026-07-21', '2026-07-25').tolist() # обратите внимание, 20 и 21 стыкуются (один остров)

# Пользователь 2: Короткие частые стрики
dates_u2 = pd.date_range('2026-07-02', '2026-07-04').tolist() + \
           pd.date_range('2026-07-08', '2026-07-09').tolist() + \
           pd.date_range('2026-07-15', '2026-07-18').tolist()

# Пользователь 3: Одиночные заходы и один длинный стрик
dates_u3 = [pd.Timestamp('2026-07-01'), pd.Timestamp('2026-07-05')] + \
           pd.date_range('2026-07-10', '2026-07-16').tolist()

for d in dates_u1:
    data.append({'user_id': 101, 'activity_date': d.date()})
for d in dates_u2:
    data.append({'user_id': 102, 'activity_date': d.date()})
for d in dates_u3:
    data.append({'user_id': 103, 'activity_date': d.date()})

# Создаем Pandas DataFrame
user_activity = pd.DataFrame(data)

# Проверим первые строки
user_activity

,user_id,activity_date
0,101,2026-07-01
1,101,2026-07-02
2,101,2026-07-03
3,101,2026-07-04
4,101,2026-07-05
5,101,2026-07-06
6,101,2026-07-07
7,101,2026-07-08
8,101,2026-07-09
9,101,2026-07-10


In [ ]:
%%sql result <<

WITH

unique_periods AS (
-- Оставляем только уникальные моменты времени с точностью дескретизации острова (в данном случае день)
-- и присваиваем каждому dt его порядковый номер.
    SELECT
        *
        , ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY dt) as rn
    FROM (
        SELECT DISTINCT  -- на случай если могут происходить разные события
            user_id
            , DATE_TRUNC('day', activity_date) AS dt
        FROM user_activity
    )
),

session_groups AS (
-- Для каждого дня вычисляем "ключ группы".
-- У последовательных дней значение (dt - rn * 1 day)
-- будет одинаковым, что позволяет объединить их в непрерывные интервалы.

    SELECT
        user_id
        , dt
        , dt - rn * INTERVAL '1 day' as grp
    FROM unique_periods
),
all_streaks AS (
    -- Для каждой непрерывной последовательности определяем её начало и конец.
    SELECT
        user_id
        --, grp
        , MIN(dt) as session_start
        , MAX(dt) as session_end
        -- Длительность стрика в днях (включая границы)
        , count(*) as length
    FROM session_groups
    GROUP BY user_id, grp
),
ranked_streaks AS (
    SELECT
    *,
    -- DENSE_RANK вернет несколько записей совпадающей длины
    DENSE_RANK() OVER (PARTITION BY user_id ORDER BY length DESC) as rank
    FROM all_streaks
)
SELECT
*
FROM ranked_streaks
WHERE rank=1

,user_id,session_start,session_end,length,rank
0,102,2026-07-15,2026-07-18,4,1
1,101,2026-07-15,2026-07-25,11,1
2,103,2026-07-10,2026-07-16,7,1


# Case - cohort retention

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Генерируем 100 пользователей, регистрирующихся в разные дни июля 2026 года
n_users = 100
user_ids = np.arange(1001, 1001 + n_users)

data = []
cohort_start = pd.Timestamp('2026-07-01')

for uid in user_ids:
    # Дата первого захода (когорта)
    first_day = cohort_start + pd.Timedelta(days=int(np.random.choice(range(5))))
    data.append({'user_id': uid, 'event_time': first_day + pd.Timedelta(hours=int(np.random.randint(0, 12)))})

    # Генерируем повторные заходы с разной вероятностью для разных дней
    for day_offset in range(1, 8):
        # Имитируем затухание Retention (чем дальше день, тем меньше вероятность возврата)
        prob = 0.6 if day_offset == 1 else (0.4 if day_offset <= 3 else 0.25)
        if np.random.rand() < prob:
            event_date = first_day + pd.Timedelta(days=day_offset, hours=int(np.random.randint(0, 23)))
            data.append({'user_id': uid, 'event_time': event_date})

user_events = pd.DataFrame(data)
user_events.head(10)

,user_id,event_time
0,1001,2026-07-04 10:00:00
1,1001,2026-07-08 10:00:00
2,1002,2026-07-05 01:00:00
3,1002,2026-07-08 20:00:00
4,1002,2026-07-11 16:00:00
5,1003,2026-07-02 11:00:00
6,1003,2026-07-04 14:00:00
7,1004,2026-07-05 08:00:00
8,1004,2026-07-06 03:00:00
9,1004,2026-07-10 01:00:00


In [ ]:
%%sql result <<

WITH
user_first_activity AS (
    SELECT DISTINCT
    user_id
    , CAST(event_time AS date) as dt
    , MIN(CAST(event_time AS date)) OVER(PARTITION BY user_id) as cohort_date
    FROM user_events
),
user_day_offsets AS (
    SELECT
    user_id
    , cohort_date
    , dt
    , dt - cohort_date AS lifetime
    FROM user_first_activity
    ORDER BY user_id
),
cohort_sizes AS (
    SELECT
        cohort_date
        , COUNT(DISTINCT user_id) AS cohort_size
    FROM user_day_offsets
    WHERE lifetime = 0
    GROUP BY cohort_date
),
retained_users AS (
    SELECT
        cohort_date
        , lifetime
        , COUNT(DISTINCT user_id) AS users
    FROM user_day_offsets
    WHERE lifetime > 0
    GROUP BY cohort_date, lifetime
)
SELECT
    cohort_sizes.cohort_date
    , retained_users.lifetime
    , cohort_sizes.cohort_size
    , retained_users.users
    , 1.0 * retained_users.users / cohort_sizes.cohort_size AS retention
FROM retained_users
LEFT JOIN cohort_sizes ON cohort_sizes.cohort_date = retained_users.cohort_date
ORDER BY cohort_sizes.cohort_date, lifetime

,cohort_date,lifetime,cohort_size,users,retention
0,2026-07-01,1,19,7,0.368421
1,2026-07-01,2,19,7,0.368421
2,2026-07-01,3,19,10,0.526316
3,2026-07-01,4,19,5,0.263158
4,2026-07-01,5,19,6,0.315789
5,2026-07-01,6,19,4,0.210526
6,2026-07-01,7,19,3,0.157895
7,2026-07-02,1,22,10,0.454545
8,2026-07-02,2,22,11,0.500000
9,2026-07-02,3,22,5,0.227273


In [ ]:
%%sql result <<

WITH
users_cohorts AS (
    SELECT DISTINCT
    user_id
    , CAST(event_time AS date) as dt
    , MIN(CAST(event_time AS date)) OVER(PARTITION BY user_id) as cohort_date
    FROM user_events
),
retention_flags AS (
    SELECT
        uc.user_id,
        uc.cohort_date,

        -- Проверяем, был ли пользователь активен на следующий день после регистрации.
        MAX(
            CASE
                WHEN ua.dt = uc.cohort_date + INTERVAL '1 day'
                THEN 1
                ELSE 0
            END
        ) AS retained_d1,

        -- Проверяем возврат на 7-й день.
        MAX(
            CASE
                WHEN ua.dt = uc.cohort_date + INTERVAL '7 day'
                THEN 1
                ELSE 0
            END
        ) AS retained_d7,

        -- Проверяем возврат на 30-й день.
        MAX(
            CASE
                WHEN ua.dt = uc.cohort_date + INTERVAL '30 day'
                THEN 1
                ELSE 0
            END
        ) AS retained_d30

    FROM users_cohorts uc

    -- LEFT JOIN важен:
    -- даже если пользователь не вернулся,
    -- он всё равно должен остаться в расчёте когорты.
    LEFT JOIN users_cohorts ua
        ON uc.user_id = ua.user_id
       AND ua.dt IN (
            uc.cohort_date + INTERVAL '1 day',
            uc.cohort_date + INTERVAL '7 day',
            uc.cohort_date + INTERVAL '30 day'
       )

    GROUP BY
        uc.user_id,
        uc.cohort_date
)

SELECT
    cohort_date,
    COUNT(*) AS cohort_size,

    SUM(retained_d1) AS retained_users_d1,
    ROUND(SUM(retained_d1)::numeric / COUNT(*), 4) AS retention_d1,

    SUM(retained_d7) AS retained_users_d7,
    ROUND(SUM(retained_d7)::numeric / COUNT(*), 4) AS retention_d7,

    SUM(retained_d30) AS retained_users_d30,
    ROUND(SUM(retained_d30)::numeric / COUNT(*), 4) AS retention_d30

FROM retention_flags
GROUP BY cohort_date
ORDER BY cohort_date;

,cohort_date,cohort_size,retained_users_d1,retention_d1,retained_users_d7,retention_d7,retained_users_d30,retention_d30
0,2026-07-01,19,7.0,0.3684,3.0,0.1579,0.0,0.0
1,2026-07-02,22,10.0,0.4545,2.0,0.0909,0.0,0.0
2,2026-07-03,23,12.0,0.5217,7.0,0.3043,0.0,0.0
3,2026-07-04,15,11.0,0.7333,9.0,0.6000,0.0,0.0
4,2026-07-05,21,10.0,0.4762,4.0,0.1905,0.0,0.0


# Case - LTV (Cumulative LTV / Cumulative ARPU)

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# 1. Генерируем 1000 пользователей, зарегистрированных с 1 по 5 июля 2026
n_users = 1000
user_ids = np.arange(10001, 10001 + n_users)
reg_dates = pd.date_range('2026-07-01', '2026-07-05')

users_data = []
for uid in user_ids:
    reg_date = np.random.choice(reg_dates)
    users_data.append({'user_id': uid, 'reg_date': reg_date})

users = pd.DataFrame(users_data)

# 2. Генерируем покупки
purchases_data = []
for _, user in users.iterrows():
    uid = user['user_id']
    rdate = user['reg_date']

    # С вероятностью 30% пользователь совершает покупку в день регистрации (Day 0)
    if np.random.rand() < 0.3:
        purchases_data.append({
            'user_id': uid,
            'purchase_time': rdate + pd.Timedelta(hours=int(np.random.randint(0, 12))),
            'amount': round(np.random.uniform(5, 20), 2)
        })

    # Повторные покупки в течение следующих 14 дней
    for day_offset in range(1, 15):
        # Вероятность покупки снижается
        if np.random.rand() < 0.08:
            purchases_data.append({
                'user_id': uid,
                'purchase_time': rdate + pd.Timedelta(days=day_offset, hours=int(np.random.randint(0, 23))),
                'amount': round(np.random.uniform(10, 50), 2)
            })

purchases = pd.DataFrame(purchases_data)

print(f"Пользователей: {len(users)}, Покупок: {len(purchases)}")
display(purchases.head(10))
display(users.head(10))

Пользователей: 1000, Покупок: 1457


,user_id,purchase_time,amount
0,10002,2026-07-12 08:00:00,18.22
1,10003,2026-07-06 18:00:00,19.93
2,10003,2026-07-09 00:00:00,24.50
3,10003,2026-07-12 20:00:00,30.61
4,10004,2026-07-05 06:00:00,13.93
5,10004,2026-07-06 22:00:00,27.07
6,10005,2026-07-08 02:00:00,24.93
7,10005,2026-07-12 17:00:00,44.68
8,10007,2026-07-14 19:00:00,34.97
9,10008,2026-07-03 03:00:00,19.62


,user_id,reg_date
0,10001,2026-07-04
1,10002,2026-07-05
2,10003,2026-07-03
3,10004,2026-07-05
4,10005,2026-07-05
5,10006,2026-07-02
6,10007,2026-07-03
7,10008,2026-07-03
8,10009,2026-07-03
9,10010,2026-07-05


$$\text{Cumulative } LTV_{day} = \frac{\sum_{t=0}^{day} \text{Revenue}_t}{\text{Cohort Size}}$$

In [ ]:
%%sql result <<

WITH
cohort_sizes AS (
    SELECT
        reg_date AS cohort_date
        , COUNT(DISTINCT user_id) AS cohort_size
    FROM users
    GROUP BY reg_date
),
user_amounts AS (
    SELECT
        purchases.user_id
        , users.reg_date AS cohort_date
        , purchase_time::DATE - cohort_date::DATE AS lifetime
        , amount
    FROM purchases
    LEFT JOIN users ON users.user_id = purchases.user_id
    ORDER BY user_id, lifetime
),
lifetime_amounts AS (
    SELECT
        cohort_date
        , lifetime
        --, SUM(amount) as amount
        , SUM(SUM(amount)) OVER (PARTITION BY cohort_date ORDER BY lifetime) as cum_amount
    FROM user_amounts
    GROUP BY cohort_date, lifetime
    ORDER BY cohort_date, lifetime
),
ltv AS (
    SELECT
        lifetime_amounts.cohort_date
        , cohort_sizes.cohort_size
        , lifetime_amounts.lifetime
        , lifetime_amounts.cum_amount / cohort_sizes.cohort_size AS LTV
    FROM lifetime_amounts
    LEFT JOIN cohort_sizes ON cohort_sizes.cohort_date = lifetime_amounts.cohort_date
    ORDER BY cohort_date, lifetime
)

SELECT * FROM ltv

,cohort_date,cohort_size,lifetime,LTV
0,2026-07-01,210,0,3.933905
1,2026-07-01,210,1,6.282286
2,2026-07-01,210,2,8.409048
3,2026-07-01,210,3,10.613619
4,2026-07-01,210,4,13.377238
...,...,...,...,...
70,2026-07-05,204,10,28.055931
71,2026-07-05,204,11,30.799608
72,2026-07-05,204,12,31.932304
73,2026-07-05,204,13,34.684461


# Case - Sessionization pattern

In [ ]:
import pandas as pd
import numpy as np

# Фиксируем seed для воспроизводимости
np.random.seed(42)

events_list = []
base_time = pd.Timestamp('2026-08-01 10:00:00')

# --- User 101: 2 четкие сессии ---
# Сессия 1 (3 клика внутри 15 минут)
t = base_time
events_list.extend([
    {'user_id': 101, 'event_time': t, 'event_type': 'page_view'},
    {'user_id': 101, 'event_time': t + pd.Timedelta(minutes=5), 'event_type': 'click'},
    {'user_id': 101, 'event_time': t + pd.Timedelta(minutes=12), 'event_type': 'add_to_cart'}
])
# Сессия 2 (пауза 45 минут > 30)
t_session2 = t + pd.Timedelta(minutes=12 + 45)
events_list.extend([
    {'user_id': 101, 'event_time': t_session2, 'event_type': 'checkout'},
    {'user_id': 101, 'event_time': t_session2 + pd.Timedelta(minutes=3), 'event_type': 'payment'}
])

# --- User 102: Пограничный случай (пауза ровно 30 минут) ---
t2 = base_time + pd.Timedelta(hours=1)
events_list.extend([
    {'user_id': 102, 'event_time': t2, 'event_type': 'page_view'},
    # Пауза ровно 30 минут (не должна создать новую сессию при > '30 minutes')
    {'user_id': 102, 'event_time': t2 + pd.Timedelta(minutes=30), 'event_type': 'click'},
    # Пауза 31 минута (должна создать новую сессию)
    {'user_id': 102, 'event_time': t2 + pd.Timedelta(minutes=61), 'event_type': 'page_view'}
])

# --- User 103: Одиночные редкие заходы (каждый заход — отдельная сессия) ---
t3 = base_time + pd.Timedelta(hours=3)
events_list.extend([
    {'user_id': 103, 'event_time': t3, 'event_type': 'page_view'},
    {'user_id': 103, 'event_time': t3 + pd.Timedelta(hours=2), 'event_type': 'page_view'},
    {'user_id': 103, 'event_time': t3 + pd.Timedelta(hours=5), 'event_type': 'page_view'}
])

events = pd.DataFrame(events_list)

# Проверяем сформированные данные
events.sort_values(by=['user_id', 'event_time'])

,user_id,event_time,event_type
0,101,2026-08-01 10:00:00,page_view
1,101,2026-08-01 10:05:00,click
2,101,2026-08-01 10:12:00,add_to_cart
3,101,2026-08-01 10:57:00,checkout
4,101,2026-08-01 11:00:00,payment
5,102,2026-08-01 11:00:00,page_view
6,102,2026-08-01 11:30:00,click
7,102,2026-08-01 12:01:00,page_view
8,103,2026-08-01 13:00:00,page_view
9,103,2026-08-01 15:00:00,page_view


In [ ]:
%%sql result <<

WITH
events_with_prev AS (
-- Шаг 1. Для каждого события находим время предыдущего события
    SELECT
        user_id,
        event_time,

        -- Предыдущее событие этого же пользователя
        LAG(event_time) OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS prev_event_time

    FROM events
),
session_flags AS (
-- Шаг 2. Определяем начало новой сессии
    SELECT
        user_id,
        event_time,

        CASE
            -- Первое событие пользователя всегда начинает новую сессию
            WHEN prev_event_time IS NULL THEN 1

            -- Если прошло больше 30 минут — начинается новая сессия
            WHEN event_time - prev_event_time > INTERVAL '30 minutes' THEN 1

            -- Иначе продолжаем текущую сессию
            ELSE 0
        END AS is_new_session

    FROM events_with_prev
)

-- Шаг 3. Нумеруем сессии
SELECT
    user_id,
    event_time,

    -- Накопительная сумма увеличивается каждый раз,
    -- когда встречается начало новой сессии
    SUM(is_new_session) OVER (
        PARTITION BY user_id
        ORDER BY event_time
    ) AS session_id

FROM session_flags

ORDER BY
    user_id,
    event_time;


,user_id,event_time,session_id
0,101,2026-08-01 10:00:00,1.0
1,101,2026-08-01 10:05:00,1.0
2,101,2026-08-01 10:12:00,1.0
3,101,2026-08-01 10:57:00,2.0
4,101,2026-08-01 11:00:00,2.0
5,102,2026-08-01 11:00:00,1.0
6,102,2026-08-01 11:30:00,1.0
7,102,2026-08-01 12:01:00,2.0
8,103,2026-08-01 13:00:00,1.0
9,103,2026-08-01 15:00:00,2.0


# Case - cume_dist and percent_rank and percentile_count

In [ ]:
import pandas as pd

# Создаем синтетический датасет чеков клиентов
df_orders = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105, 106, 107, 108],
    "order_amount": [150, 300, 300, 500, 800, 1200, 1200, 2500]
})

# Проверяем структуру
df_orders

,customer_id,order_amount
0,101,150
1,102,300
2,103,300
3,104,500
4,105,800
5,106,1200
6,107,1200
7,108,2500


In [ ]:
%%sql
SELECT
    customer_id,
    order_amount,

    -- Порядковый номер и ранг для наглядности
    ROW_NUMBER() OVER (ORDER BY order_amount) AS row_num,
    RANK() OVER (ORDER BY order_amount) AS rank_val,

    -- CUME_DIST: Доля элементов <= текущего (от 1/N до 1.0)
    ROUND(CUME_DIST() OVER (ORDER BY order_amount), 3) AS cume_dist_val,

    -- PERCENT_RANK: Относительный ранг (от 0.0 до 1.0)
    ROUND(PERCENT_RANK() OVER (ORDER BY order_amount), 3) AS percent_rank_val

FROM df_orders
ORDER BY order_amount;

,customer_id,order_amount,row_num,rank_val,cume_dist_val,percent_rank_val
0,101,150,1,1,0.125,0.000
1,102,300,2,2,0.375,0.143
2,103,300,3,2,0.375,0.143
3,104,500,4,4,0.500,0.429
4,105,800,5,5,0.625,0.571
5,106,1200,6,6,0.875,0.714
6,107,1200,7,6,0.875,0.714
7,108,2500,8,8,1.000,1.000


In [ ]:
%%sql
SELECT
    --customer_id,

    PERCENTILE_CONT(0.5)
        WITHIN GROUP (ORDER BY order_amount) AS median,

    PERCENTILE_CONT(0.9)
        WITHIN GROUP (ORDER BY order_amount) AS p90,

    PERCENTILE_CONT(0.95)
        WITHIN GROUP (ORDER BY order_amount) AS p95

FROM df_orders
--GROUP BY customer_id ORDER BY customer_id;

,median,p90,p95
0,650.0,1590.0,2045.0


# Case - Moving average / rolling window

In [ ]:
import pandas as pd
import numpy as np

# Фиксируем seed для воспроизводимости
np.random.seed(42)

# Генерируем даты с 1 по 31 июля 2026 года (31 день)
dates = pd.date_range('2026-07-01', '2026-07-31', freq='D')

# Формируем дневную выручку: База + Недельная сезонность + Тренд + Шум
base_revenue = 100000
data = []

for i, date in enumerate(dates):
    # Тренд: небольшое увеличение со временем
    trend = i * 1500

    # Сезонность: в выходные (суббота/воскресенье) выручка выше на 40%
    day_of_week = date.dayofweek
    weekend_multiplier = 1.4 if day_of_week in [5, 6] else 1.0

    # Случайные колебания (шум ±10%)
    noise = np.random.uniform(0.9, 1.1)

    day_revenue = round((base_revenue + trend) * weekend_multiplier * noise, 2)

    data.append({
        'sale_date': date.strftime('%Y-%m-%d'),
        'revenue': day_revenue
    })

daily_sales = pd.DataFrame(data)

# Проверяем первые 10 дней
daily_sales.head(10)

,sale_date,revenue
0,2026-07-01,97490.80
1,2026-07-02,110649.50
2,2026-07-03,107779.08
3,2026-07-04,149186.75
4,2026-07-05,138190.63
5,2026-07-06,100103.88
6,2026-07-07,99366.22
7,2026-07-08,118592.49
8,2026-07-09,114264.98
9,2026-07-10,118223.25


In [ ]:
%%sql
SELECT
    sale_date,
    revenue,

    CAST(
    -- Средняя выручка за текущий день и шесть предыдущих дней
    AVG(revenue) OVER (
        ORDER BY sale_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    )
    AS DECIMAL(16,2))
    AS revenue_ma_7d

FROM daily_sales

ORDER BY sale_date;

,sale_date,revenue,revenue_ma_7d
0,2026-07-01,97490.80,97490.80
1,2026-07-02,110649.50,104070.15
2,2026-07-03,107779.08,105306.46
3,2026-07-04,149186.75,116276.53
4,2026-07-05,138190.63,120659.35
5,2026-07-06,100103.88,117233.44
6,2026-07-07,99366.22,114680.98
7,2026-07-08,118592.49,117695.51
8,2026-07-09,114264.98,118212.00
9,2026-07-10,118223.25,119704.03


#  Case - Funnel analysis A → B → C

In [ ]:
import pandas as pd

# Формируем события с разным порядком и полнотой прохождения шагов A, B, C
events_data = [
    # User 1: Идеальное прохождение A -> B -> C
    {"user_id": 1, "event_name": "A", "event_time": "2026-08-01 10:00:00"},
    {"user_id": 1, "event_name": "B", "event_time": "2026-08-01 10:05:00"},
    {"user_id": 1, "event_name": "C", "event_time": "2026-08-01 10:15:00"},

    # User 2: Нарушен порядок (B раньше A) -> B -> A -> C
    {"user_id": 2, "event_name": "B", "event_time": "2026-08-01 11:00:00"},
    {"user_id": 2, "event_name": "A", "event_time": "2026-08-01 11:10:00"},
    {"user_id": 2, "event_name": "C", "event_time": "2026-08-01 11:20:00"},

    # User 3: Отвал на шаге C (сделал только A -> B)
    {"user_id": 3, "event_name": "A", "event_time": "2026-08-01 12:00:00"},
    {"user_id": 3, "event_name": "B", "event_time": "2026-08-01 12:02:00"},

    # User 4: Успешный проход с дублями событий (MIN выберет первичное время)
    {"user_id": 4, "event_name": "A", "event_time": "2026-08-01 13:00:00"},
    {"user_id": 4, "event_name": "A", "event_time": "2026-08-01 13:01:00"},  # Повторный шаг A
    {"user_id": 4, "event_name": "B", "event_time": "2026-08-01 13:10:00"},
    {"user_id": 4, "event_name": "C", "event_time": "2026-08-01 13:25:00"},

    # User 5: Сделал только один шаг A
    {"user_id": 5, "event_name": "A", "event_time": "2026-08-01 14:00:00"},
]

events = pd.DataFrame(events_data)

# Приводим event_time к формату datetime
events['event_time'] = pd.to_datetime(events['event_time'])

events

,user_id,event_name,event_time
0,1,A,2026-08-01 10:00:00
1,1,B,2026-08-01 10:05:00
2,1,C,2026-08-01 10:15:00
3,2,B,2026-08-01 11:00:00
4,2,A,2026-08-01 11:10:00
5,2,C,2026-08-01 11:20:00
6,3,A,2026-08-01 12:00:00
7,3,B,2026-08-01 12:02:00
8,4,A,2026-08-01 13:00:00
9,4,A,2026-08-01 13:01:00


In [ ]:
%%sql
WITH user_steps AS (
    SELECT
        user_id,
        -- Первое время, когда пользователь сделал шаг A
        MIN(CASE WHEN event_name = 'A' THEN event_time END) AS step_a_time,
        -- Первое время, когда пользователь сделал шаг B
        MIN(CASE WHEN event_name = 'B' THEN event_time END) AS step_b_time,
        -- Первое время, когда пользователь сделал шаг C
        MIN(CASE WHEN event_name = 'C' THEN event_time END) AS step_c_time
    FROM events
    GROUP BY user_id
)

SELECT
    user_id,
    step_a_time,
    step_b_time,
    step_c_time
FROM user_steps
WHERE
    -- Пользователь должен пройти все три шага
    step_a_time IS NOT NULL
    AND step_b_time IS NOT NULL
    AND step_c_time IS NOT NULL

    -- И шаги должны идти в правильном порядке: A → B → C
    AND step_a_time < step_b_time
    AND step_b_time < step_c_time;

,user_id,step_a_time,step_b_time,step_c_time
0,4,2026-08-01 13:00:00,2026-08-01 13:10:00,2026-08-01 13:25:00
1,1,2026-08-01 10:00:00,2026-08-01 10:05:00,2026-08-01 10:15:00


In [ ]:
%%sql

SELECT
    user_id,
    STRING_AGG(event_name, ' → ' ORDER BY event_time) AS source_path
FROM events
GROUP BY user_id ORDER BY user_id;

,user_id,source_path
0,1,A → B → C
1,2,B → A → C
2,3,A → B
3,4,A → A → B → C
4,5,A


# Case - Anti-join / finding missing records

In [ ]:
import pandas as pd

# 1. Таблица пользователей
users = pd.DataFrame({
    "user_id": [101, 102, 103, 104, 105],
    "registration_date": [
        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-04",
        "2026-07-05"
    ]
})

# 2. Таблица заказов (пользователи 103 и 105 заказов НЕ делали)
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4],
    "user_id": [101, 101, 102, 104],
    "order_date": [
        "2026-07-01 14:20:00",
        "2026-07-03 10:15:00",
        "2026-07-02 18:00:00",
        "2026-07-05 09:30:00"
    ],
    "amount": [1500, 2300, 850, 4100]
})

# Вывод исходных таблиц

display(users)
display(orders)

,user_id,registration_date
0,101,2026-07-01
1,102,2026-07-02
2,103,2026-07-03
3,104,2026-07-04
4,105,2026-07-05


,order_id,user_id,order_date,amount
0,1,101,2026-07-01 14:20:00,1500
1,2,101,2026-07-03 10:15:00,2300
2,3,102,2026-07-02 18:00:00,850
3,4,104,2026-07-05 09:30:00,4100


In [ ]:
%%sql

-- Находим пользователей, которые зарегистрировались,
-- но ни разу не сделали заказ

SELECT
    u.user_id,
    u.registration_date
FROM users AS u

-- Пытаемся найти заказ каждого пользователя
LEFT JOIN orders AS o
    ON u.user_id = o.user_id

-- Если заказ не найден,
-- поля из таблицы orders будут NULL
WHERE o.user_id IS NULL;

,user_id,registration_date
0,103,2026-07-03
1,105,2026-07-05


# Case - Deduplication with ROW_NUMBER pattern

In [ ]:
import pandas as pd

# Создаем синтетический датасет событий пользователей
user_events = pd.DataFrame({
    "user_id": [101, 101, 101, 102, 103, 103],
    "event_time": [
        "2026-08-01 10:00:00",
        "2026-08-01 10:15:00",
        "2026-08-01 10:05:00", # Для 101 самое свежее — 10:15:00
        "2026-08-01 11:30:00", # У 102 всего 1 событие
        "2026-08-01 12:00:00",
        "2026-08-01 12:45:00"  # Для 103 самое свежее — 12:45:00
    ],
    "event_name": [
        "page_view",
        "checkout",
        "add_to_cart",
        "page_view",
        "login",
        "logout"
    ]
})

# Приводим event_time к формату datetime
user_events["event_time"] = pd.to_datetime(user_events["event_time"])

user_events

,user_id,event_time,event_name
0,101,2026-08-01 10:00:00,page_view
1,101,2026-08-01 10:15:00,checkout
2,101,2026-08-01 10:05:00,add_to_cart
3,102,2026-08-01 11:30:00,page_view
4,103,2026-08-01 12:00:00,login
5,103,2026-08-01 12:45:00,logout


In [ ]:
%%sql

WITH ranked_events AS (

    SELECT
        user_id,
        event_time,
        event_name,

        -- Нумеруем события отдельно для каждого пользователя.
        -- Самое новое событие получает номер 1.
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY event_time DESC
        ) AS rn

    FROM user_events
)

SELECT
    user_id,
    event_time,
    event_name

FROM ranked_events
-- Оставляем только последнее событие пользователя.
WHERE rn = 1
ORDER BY user_id;

,user_id,event_time,event_name
0,101,2026-08-01 10:15:00,checkout
1,102,2026-08-01 11:30:00,page_view
2,103,2026-08-01 12:45:00,logout


# Case - lead/lag

In [ ]:
import pandas as pd

# Создаем синтетический датасет транзакций
transactions_data = [
    # User 101: Регулярные покупки, затем резкий скачок (аномалия)
    {"transaction_id": 1, "user_id": 101, "created_at": "2026-08-01 10:00:00", "amount": 1000.0},
    {"transaction_id": 2, "user_id": 101, "created_at": "2026-08-02 14:30:00", "amount": 1200.0},
    {"transaction_id": 3, "user_id": 101, "created_at": "2026-08-03 11:15:00", "amount": 3500.0}, # Аномалия (> 2x от 1200)
    {"transaction_id": 4, "user_id": 101, "created_at": "2026-08-05 09:00:00", "amount": 1100.0},

    # User 102: Покупки с большими перерывами
    {"transaction_id": 5, "user_id": 102, "created_at": "2026-08-01 12:00:00", "amount": 5000.0},
    {"transaction_id": 6, "user_id": 102, "created_at": "2026-08-04 18:20:00", "amount": 4800.0},

    # User 103: Всего одна покупка
    {"transaction_id": 7, "user_id": 103, "created_at": "2026-08-02 16:45:00", "amount": 2500.0},
]

transactions = pd.DataFrame(transactions_data)
transactions["created_at"] = pd.to_datetime(transactions["created_at"])

transactions

,transaction_id,user_id,created_at,amount
0,1,101,2026-08-01 10:00:00,1000.0
1,2,101,2026-08-02 14:30:00,1200.0
2,3,101,2026-08-03 11:15:00,3500.0
3,4,101,2026-08-05 09:00:00,1100.0
4,5,102,2026-08-01 12:00:00,5000.0
5,6,102,2026-08-04 18:20:00,4800.0
6,7,103,2026-08-02 16:45:00,2500.0


In [ ]:
%%sql
SELECT
    user_id,
    transaction_id,
    created_at,
    amount,

    -- 1. Предыдущая сумма
    LAG(amount) OVER (
        PARTITION BY user_id
        ORDER BY created_at
    ) AS prev_amount,

    -- 2. Следующая сумма
    LEAD(amount) OVER (
        PARTITION BY user_id
        ORDER BY created_at
    ) AS next_amount,

    -- 3. Дней с предыдущей покупки (в DuckDB разница дат дает количество дней)
    created_at::DATE - LAG(created_at) OVER (
        PARTITION BY user_id
        ORDER BY created_at
    )::DATE AS days_since_prev,

    -- 4. Флаг аномалии (> 2x от предыдущей)
    CASE
        WHEN amount > 2 * LAG(amount) OVER (PARTITION BY user_id ORDER BY created_at)
            THEN 1
        ELSE 0
    END AS is_anomaly

FROM transactions
ORDER BY user_id, created_at;

,user_id,transaction_id,created_at,amount,prev_amount,next_amount,days_since_prev,is_anomaly
0,101,1,2026-08-01 10:00:00,1000.0,NaN,1200.0,<NA>,0
1,101,2,2026-08-02 14:30:00,1200.0,1000.0,3500.0,1,0
2,101,3,2026-08-03 11:15:00,3500.0,1200.0,1100.0,1,1
3,101,4,2026-08-05 09:00:00,1100.0,3500.0,NaN,2,0
4,102,5,2026-08-01 12:00:00,5000.0,NaN,4800.0,<NA>,0
5,102,6,2026-08-04 18:20:00,4800.0,5000.0,NaN,3,0
6,103,7,2026-08-02 16:45:00,2500.0,NaN,NaN,<NA>,0
